# MAESTER: the model inputs

The two inputs of the CloneVI fit of the MAESTER sample BPDCN712 (Fig. 3), built from the files of GEO GSE182685
(`preprocess/README.md`):

| output | made from |
|---|---|
| `rna_mgv_counts.h5ad`: mtDNA coverage (`X`), mutant counts and VAF of every position in the 9,442 cells of maegatk | the maegatk counts of GSM5534706, exported by `export_maegatk.R` |
| `rna_exp.h5ad`: raw counts of 18,979 genes in 9,346 cells (`raw.X`; log-normalized in `X`) and the cell metadata of the source study | the expression and metadata of GSM5534705 |

For the study, `rna_exp.h5ad` was converted from the Seurat object of the source study (`BPDCN712_Seurat_with_TCR.rds`),
which its authors distributed on a web site that is no longer online. The GEO files hold its counts and metadata
except the columns added by the source study's T/NK and TCR analyses, among them the mtDNA clones (`MT_clone`,
`MT_clone_num`); `retrain/prep_maester.py` uses `MT_clone_num` only to color a plot drawn after the fit, so these
columns are taken from the deposited file. The files are written to `OUT` and compared with the deposited ones
(`data/h5ad_files_RNA/`). Run from `preprocess/MAESTER/`.

In [1]:
import os

import h5py
import numpy as np
import pandas as pd
import scipy.io as sio
import scipy.sparse as sp
import anndata as ad

In [2]:
M = '../../MAESTER'                                     # GEO files and the output of export_maegatk.R
DEP = '../../data/h5ad_files_RNA'                       # deposited model inputs
OUT = '../../data/h5ad_files_RNA/rebuilt'
os.makedirs(OUT, exist_ok=True)

## 1. mtDNA input

The steps of the study's notebook (`MAESTER_RNAmgv2h5ad.ipynb`): for each position the mutant count is the coverage
minus the reads of the reference allele, and the VAF is the mutant count divided by the coverage (0 where there is no
coverage).

In [3]:
datadir = f'{M}/rds_output'
matname_counts = ['A_counts_fw', 'A_counts_rev', 'C_counts_fw', 'C_counts_rev', 'G_counts_fw', 'G_counts_rev', 'T_counts_fw', 'T_counts_rev', 'coverage']

genes = pd.read_csv(os.path.join(datadir, 'genes.tsv'), sep='\t')
genes.index = genes.index.astype(str)
barcodes = np.loadtxt(os.path.join(datadir, 'barcodes.tsv'), dtype='str')
mtx = sio.mmread(os.path.join(datadir, 'coverage.mtx')).toarray().T
obs = pd.DataFrame({'barcode': barcodes}, index=np.arange(len(barcodes)).astype(str))
adata_mg_c = ad.AnnData(X=mtx, obs=obs, var=genes)
for y in range(len(matname_counts)):
    adata_mg_c.layers[matname_counts[y]] = sio.mmread(os.path.join(datadir, matname_counts[y] + '.mtx')).toarray().T
adata_mg_c.var_names_make_unique()
adata_mg_c.var = adata_mg_c.var.drop(columns='strand')
adata_mg_c

AnnData object with n_obs × n_vars = 9442 × 16569
    obs: 'barcode'
    var: 'seqnames', 'start', 'end', 'width', 'refAllele'
    layers: 'A_counts_fw', 'A_counts_rev', 'C_counts_fw', 'C_counts_rev', 'G_counts_fw', 'G_counts_rev', 'T_counts_fw', 'T_counts_rev', 'coverage'

In [4]:
# positions whose reference allele is not A, C, G or T
not_A_index = np.array(adata_mg_c.var.reset_index().query('refAllele!="A"').index)
not_C_index = np.array(adata_mg_c.var.reset_index().query('refAllele!="C"').index)
not_G_index = np.array(adata_mg_c.var.reset_index().query('refAllele!="G"').index)
not_T_index = np.array(adata_mg_c.var.reset_index().query('refAllele!="T"').index)

all_mut = 0
for base, not_index in [('A', not_A_index), ('C', not_C_index), ('G', not_G_index), ('T', not_T_index)]:
    variant = adata_mg_c.X - adata_mg_c.layers[f'{base}_counts_fw'] - adata_mg_c.layers[f'{base}_counts_rev']
    variant[:, not_index] = 0
    all_mut = all_mut + variant
with np.errstate(divide='ignore', invalid='ignore'):
    VAF = np.nan_to_num(all_mut / adata_mg_c.X)

adata_mg_c_VAF = ad.AnnData(X=adata_mg_c.X, obs=adata_mg_c.obs, var=adata_mg_c.var)
adata_mg_c_VAF.layers['all_mut'] = sp.csr_matrix(all_mut)
adata_mg_c_VAF.layers['VAF'] = sp.csr_matrix(VAF)
del adata_mg_c, all_mut, VAF, variant
adata_mg_c_VAF.write_h5ad(os.path.join(OUT, 'rna_mgv_counts.h5ad'), compression='gzip')
adata_mg_c_VAF

AnnData object with n_obs × n_vars = 9442 × 16569
    obs: 'barcode'
    var: 'seqnames', 'start', 'end', 'width', 'refAllele'
    layers: 'all_mut', 'VAF'

Compared with the deposited input:

In [5]:
dep = ad.read_h5ad(f'{DEP}/rna_mgv_counts.h5ad')
new = adata_mg_c_VAF
pd.Series({'cells and obs': dep.obs.equals(new.obs),
           'positions and reference alleles': dep.var.astype(str).equals(new.var.astype(str)),
           'coverage': np.array_equal(dep.X, new.X),
           'mutant counts': (dep.layers['all_mut'] != new.layers['all_mut']).nnz == 0,
           'VAF': (dep.layers['VAF'] != new.layers['VAF']).nnz == 0})

cells and obs                      True
positions and reference alleles    True
coverage                           True
mutant counts                      True
VAF                                True
dtype: bool

In [6]:
del dep, new, adata_mg_c_VAF

## 2. Expression input

The raw counts and the metadata of GEO, with the counts log-normalized as Seurat's `NormalizeData` does (scale factor
10,000), the layout of the Seurat object converted for the study:

In [7]:
expr = pd.read_csv(f'{M}/GSM5534705_BPDCN712_10x_scRNAseq_Expression.txt.gz', sep='\t', index_col=0)
meta = pd.read_csv(f'{M}/GSM5534705_BPDCN712_10x_scRNAseq_MetaData.txt.gz', sep='\t', index_col=0)
expr.shape, meta.shape

((18979, 9346), (9346, 22))

In [8]:
counts = sp.csr_matrix(expr.to_numpy(dtype=np.float64).T)
obs = meta.loc[expr.columns].copy()
for c in obs.columns[obs.dtypes == object]:                 # text columns stored as categories
    obs[c] = obs[c].astype('category')
var = pd.DataFrame(index=expr.index.astype(str))
lognorm = sp.diags(1e4 / np.asarray(counts.sum(axis=1)).ravel()) @ counts
lognorm.data = np.log1p(lognorm.data)

rna = ad.AnnData(X=sp.csr_matrix(lognorm), obs=obs, var=var)
rna.raw = ad.AnnData(X=counts, obs=obs, var=var)
rna.write_h5ad(f'{OUT}/rna_exp.h5ad')
del expr, counts, lognorm
rna

AnnData object with n_obs × n_vars = 9346 × 18979
    obs: 'nCount_RNA', 'nFeature_RNA', 'replicate', 'CellType', 'project.umap.x', 'project.umap.y', 'TET2.S792*', 'TET2.Q1034*', 'TET2.R1216*', 'TET2.H1380Y', 'ACAP2.L97M', 'ASXL1.G642fs', 'CWF19L2.K221R', 'DOLPP1.R227S', 'HNRNPUL1.F559F', 'MAP4K5.P667S', 'RAB9A.3pUTR', 'MTAP.rearr', 'BPDCN_Tumor_score', 'Cycle_score', 'UMAP_1', 'UMAP_2'

Compared with the deposited input. Its metadata was written by SeuratDisk, which stores each factor as codes and its
levels separately; the codes are mapped to the levels to compare:

In [9]:
dep = ad.read_h5ad(f'{DEP}/rna_exp.h5ad')
with h5py.File(f'{DEP}/rna_exp.h5ad', 'r') as f:
    levels = {k: [x.decode() if isinstance(x, bytes) else str(x) for x in f['obs/__categories'][k][()]]
              for k in f['obs/__categories']}
dep_obs = dep.obs.copy()
for k, lv in levels.items():
    dep_obs[k] = [lv[i] if i >= 0 else np.nan for i in dep_obs[k].astype(int)]

def same_column(c):
    a, b = dep_obs[c], rna.obs[c]
    if pd.api.types.is_numeric_dtype(b):
        return np.allclose(a.astype(float), b.astype(float), equal_nan=True)
    return (a.astype(str).to_numpy() == b.astype(str).to_numpy()).all()

pd.Series({'cells': np.array_equal(dep.obs_names, rna.obs_names),
           'genes': np.array_equal(dep.var_names, rna.var_names),
           'raw counts (raw.X)': (dep.raw.X != rna.raw.X).nnz == 0,
           'log-normalized (X)': np.allclose(dep.X.toarray(), rna.X.toarray()),
           'metadata of GEO (%d columns)' % meta.shape[1]: all(same_column(c) for c in meta.columns)})

cells                           True
genes                           True
raw counts (raw.X)              True
log-normalized (X)              True
metadata of GEO (22 columns)    True
dtype: bool

The columns of the deposited file that GEO does not hold (the source study's T/NK reclustering, TCR and mtDNA clones):

In [10]:
missing = [c for c in dep.obs.columns if c not in rna.obs.columns]
dep_obs[missing].describe(include='all').loc[['count', 'unique']]

,UMAP_TNK_1,UMAP_TNK_2,TNK_CellType,TRB_CDR3,TRA_CDR3,MT_clone,MT_clone_num
count,8615.0,8615.0,8615,3326,1578,9346,9346.0
unique,NaN,NaN,8,32,24,24,NaN
